# 05 — Training Results: Per-Class F1 Score

## Purpose
Parse the training log produced by `scripts/train_gru_transformer.py`
and plot the per-class F1 score trajectory over training epochs.

## Prerequisite
Run `scripts/train_gru_transformer.py` first to generate the training log.
The expected log file is `../gru_transformer_out.log`
(one level up from this notebook, at the repository root).

## Input files
- `../gru_transformer_out.log` — training log with per-epoch confusion matrices

## Output files
- `../results/f1_per_class.pdf` — per-class F1 curve over 100 epochs

## What this notebook does
1. Reads the log file and extracts the confusion matrix for each epoch
2. Computes per-class precision, recall, and F1 from each confusion matrix
3. Plots the F1 trajectory for all 8 stimulus classes as a function of epoch

In [ ]:
import re
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Locate the training log. Tries the repo root first (common when running
# Jupyter from the repo root), then falls back to one level up (when running
# from the notebooks/ directory). Adjust LOG_PATH if your setup differs.
import os

LOG_CANDIDATES = [
    "gru_transformer_out.log",       # CWD = repo root
    "../gru_transformer_out.log",    # CWD = notebooks/
]
log_path = next((p for p in LOG_CANDIDATES if os.path.exists(p)), LOG_CANDIDATES[0])

with open(log_path) as f:
    text = f.read()

# Extract all confusion matrix blocks from the log.
# Each block is a sequence of lines containing only digits and spaces.
blocks = re.findall(
    r"Confusion matrix:\n((?:\s*\[[\d\s]+\]\n?)+)",
    text
)

cms = []
for block in blocks:
    rows = re.findall(r"\[([\d\s]+)\]", block)
    cms.append([[int(x) for x in r.split()] for r in rows])

cms = np.array(cms)  # shape: (n_epochs, n_classes, n_classes)

In [6]:
# cms[e, i, j] = count true=i, pred=j
tp = np.diagonal(cms, axis1=1, axis2=2)                 # (E, 8)
per_class_support = cms.sum(axis=2)                     # rows, (E, 8)
per_class_pred = cms.sum(axis=1)                        # cols, (E, 8)

recall = tp / per_class_support                         # per-class recall
precision = tp / per_class_pred                         # per-class precision

f1 = 2 * precision * recall / (precision + recall)      # (E, 8)
macro_f1 = np.nanmean(f1, axis=1)                       # one value per epoch


In [ ]:
class_names = [
    "nat_movie_one_shuff", "spontaneous", "gabors",
    "nat_movie_one_re", "drift_gratings_75_re",
    "drift_gratings_con", "dot_motion", "flashes",
]

fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(class_names):
    ax.plot(f1[:, i], label=name)

ax.plot(macro_f1, "k--", linewidth=2, label="macro avg")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1 Score — GRU-Transformer (8-class)")
ax.legend(loc="lower right", fontsize=8)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Save to results/ at the repository root
results_dir = os.path.join(os.path.dirname(os.getcwd()), "results")
os.makedirs(results_dir, exist_ok=True)
out_path = os.path.join(results_dir, "f1_per_class.pdf")
fig.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()